<a href="https://colab.research.google.com/github/kameda-yoshinari/DataAlgo-UT/blob/main/DataAlgo2026_R10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# データ構造とアルゴリズム 遅延提出（第10週） (2026/06/24　ver.A)

---


★下記３要素を**必ず直接書き換えてから**提出すること．

* 学籍番号： 202599001
* 氏名： 筑波 翼
* Colabアカウント： tsukuba.tsubasa.dummy@gmail.com

**学生同士で教えた・教わった・グループワークをした場合，必ず下記の当該行を直接書き換えて記述すること．**
申告があった場合は，論述などで内容が似ていたり，プログラムが似ていても減点はしない．（コピペレベルの同一文などは処罰対象）
**教えた側は加点対象となる．**

**[教えた側]**
教えた相手：＜氏名＞　＜学籍番号＞
（何名いてもよい；教えた相手の内容が浅くなってないか確認すること）

**[教わった側]**
教わった相手：＜氏名＞　＜学籍番号＞
（何名いてもよい；必ず教えた側に自分の名前を書いてもらうこと）

**[グループワーク]**
一緒に行った相手：＜氏名＞　＜学籍番号＞
（何名いてもよいが，必ずお互いの名前を全員記すこと）

---
# 必須課題10A FPTASの概念説明

本授業内容に即して，FPTASアルゴリズムが有する性質について述べよ．
精度保証と計算量との関係については授業内での解説レベルで丁寧に説明すること．
その後，0-1ナップサック問題におけるFPTAS近似アルゴリズムについて，精度と計算量がどのような関係にあるか具体的に述べよ（導出は必要ない）




**Background.** A $\alpha$-approximation algorithm guarantees that its output is within a factor of $\alpha$ of the optimal. The lecture defined three increasingly strong families based on how $\alpha$ relates to the input size and an accuracy parameter $\varepsilon > 0$.

**PTAS (Polynomial-Time Approximation Scheme).** $\alpha = 1 + \varepsilon$ for any $\varepsilon > 0$ chosen by the user, and the running time is polynomial in the input size $n$ for each fixed $\varepsilon$. But the running time can grow *exponentially* in $1/\varepsilon$, so making the approximation tighter becomes very expensive very quickly.

**FPTAS (Fully Polynomial-Time Approximation Scheme).** Same setup as PTAS, but the running time is polynomial in **both** $n$ **and** $1/\varepsilon$. This is the strongest of the three: you can make the answer as accurate as you want *and* the cost only grows polynomially as you tighten the bound.

So the key takeaway from lecture is the relationship:

- PTAS: polynomial in $n$, possibly exponential in $1/\varepsilon$.
- FPTAS: polynomial in $n$ and in $1/\varepsilon$ simultaneously.

FPTAS algorithms are rare and valuable. They give the user a **direct knob** — pick $\varepsilon$, pay a polynomial price, get an answer guaranteed within $1 - \varepsilon$ (for max problems) or $1 + \varepsilon$ (for min problems) of the optimum.

**0-1 knapsack FPTAS — concrete relationship.**

For the FPTAS shown in §8.2 (the rounded-value dynamic-programming approach):

- **Accuracy:** $V(\{X_a\}) \;\geq\; (1 - \varepsilon)\, V(\{X_{\text{opt}}\})$. The approximate value is within a factor of $1 - \varepsilon$ of the true optimum, no matter how small you make $\varepsilon$.
- **Time complexity:** $O\!\left(\dfrac{n^3}{\varepsilon}\right)$, polynomial in both $n$ and $1/\varepsilon$.

So if you want a 1%-accurate answer ($\varepsilon = 0.01$), you pay roughly $100 \cdot n^3$ operations. If you want a 0.1%-accurate answer ($\varepsilon = 0.001$), you pay roughly $1000 \cdot n^3$. The cost grows only linearly with the inverse-accuracy parameter — exactly what makes this a *fully* polynomial scheme.

---
# 必須課題10B ミーリーマシンのファイルからの読み込み

ミーリーマシンを定義するテキストファイルを用意し，それを読み込んで実行するCプログラムを作成せよ．
入力文字・出力文字として考えるのは0-9の数字のみでよい．
状態は0以上の整数で表すこと．
受理状態については考慮しなくてよい．
用意するテキストファイルの形式については各自で考えてよいが，解説をつけること．
対象とするミーリーマシンは，授業内で用いていたものから利用してよい．
（他のミーリーマシンでもよいが，その場合は，そのミーリーマシンがどういう状態遷移を表現しているものか説明を用意すること）





**Mealy machine I am using.** I picked the **1-clock-delay** Mealy machine from the lecture (`FA-mealy.c`). It reads a stream of `0`/`1` digits and outputs each digit one step later. The first output is always `0` (because there is no previous input yet).

**File format.**

The file is a plain text file with three blocks of whitespace-separated integers. Lines starting with `#` are comments (ignored). The reader skips blank lines.

1. **Header line** — three integers: `NUM_STATE NUM_INPUTLETTERS INIT_STATE`.
2. **Transition table** — `NUM_STATE` rows, each with `NUM_INPUTLETTERS` integers. Row $i$, column $j$ = the next state when the machine is in state $i$ and reads input letter $j$.
3. **Output table** — same shape as the transition table. Row $i$, column $j$ = the output letter emitted when the machine is in state $i$ and reads input letter $j$.

This format is simple to read with `fscanf("%d", ...)`. The order is fixed: header, then all transitions, then all outputs. The reader uses `fscanf` in a loop and ignores `#`-comments and blank lines.

(The file `mealy1clk.txt` in the next cell encodes the 1-clock-delay machine — 3 states, 2 input letters, init state 0. The transition and output tables are the same numbers as the lecture program's `ttable` and `otable`.)

In [ ]:
%%writefile mealy1clk.txt
# 1-clock-delay Mealy machine (from the lecture FA-mealy.c).
# Reads 0/1 input symbols and outputs each symbol one step later.
# First output is always 0 (no previous input).
#
# Format:
#   header line:       NUM_STATE  NUM_INPUTLETTERS  INIT_STATE
#   transition table:  NUM_STATE rows of NUM_INPUTLETTERS integers
#   output table:      NUM_STATE rows of NUM_INPUTLETTERS integers

3 2 0

# transition table (ttable[state][input] = next state)
1 2
1 2
1 2

# output table (otable[state][input] = output letter)
0 0
0 0
1 1


In [ ]:
%%writefile FA-mealy-file.c
// Mealy machine that loads its definition from a text file.
// Based on FA-mealy.c from DataAlgo_UT(018)_FiniteAutomaton.
// Usage: ./FA-mealy-file <machine_file.txt>
//   Then type digits 0-9 on stdin; the machine prints the output letter
//   for each input. Ctrl-D (EOF) to quit.
#include <stdio.h>
#include <stdlib.h>
#include <ctype.h>

// Maximum machine size we can handle.
// (Generous — the actual machine size is loaded from the file.)
#define MAX_STATE 256
#define MAX_LETTER 10

int NUM_STATE = 0;
int NUM_INPUTLETTERS = 0;
int INITSTATE = 0;

int ttable[MAX_STATE][MAX_LETTER];
int otable[MAX_STATE][MAX_LETTER];

// Read one integer from fd, skipping whitespace and '#'-comment lines.
static int read_int(FILE *fd, int *out) {
    int c;
    while ((c = fgetc(fd)) != EOF) {
        if (c == '#') {
            // skip to end of line
            while ((c = fgetc(fd)) != EOF && c != '\n') { }
            continue;
        }
        if (isspace(c)) continue;
        ungetc(c, fd);
        if (fscanf(fd, "%d", out) == 1) return 0;
        return -1;
    }
    return -1;  // EOF before any integer
}

int load_machine(const char *path) {
    FILE *fd = fopen(path, "r");
    if (!fd) { perror(path); return -1; }

    // Header
    if (read_int(fd, &NUM_STATE) || read_int(fd, &NUM_INPUTLETTERS)
        || read_int(fd, &INITSTATE)) {
        fprintf(stderr, "Failed to read header.\n");
        fclose(fd); return -2;
    }
    if (NUM_STATE <= 0 || NUM_STATE > MAX_STATE
        || NUM_INPUTLETTERS <= 0 || NUM_INPUTLETTERS > MAX_LETTER
        || INITSTATE < 0 || INITSTATE >= NUM_STATE) {
        fprintf(stderr, "Header values out of range.\n");
        fclose(fd); return -3;
    }

    // Transition table
    for (int s = 0; s < NUM_STATE; s++)
        for (int l = 0; l < NUM_INPUTLETTERS; l++)
            if (read_int(fd, &ttable[s][l])) {
                fprintf(stderr, "Bad transition entry at (%d,%d).\n", s, l);
                fclose(fd); return -4;
            }

    // Output table
    for (int s = 0; s < NUM_STATE; s++)
        for (int l = 0; l < NUM_INPUTLETTERS; l++)
            if (read_int(fd, &otable[s][l])) {
                fprintf(stderr, "Bad output entry at (%d,%d).\n", s, l);
                fclose(fd); return -5;
            }

    fclose(fd);
    return 0;
}

// Run the machine on stdin (same idea as FA-mealy.c).
int mealymachine(int ss) {
    int c, t, cs, ns;
    cs = ss;
    printf("Input Output    : 1st  input => ");
    while ((c = getc(stdin)) != EOF) {
        t = c - '0';
        if (t >= 0 && t < NUM_INPUTLETTERS) {
            ns = ttable[cs][t];
            printf("    %d      %d    : next input => ", t, otable[cs][t]);
            cs = ns;
        }
    }
    return 0;
}

int main(int argc, char *argv[]) {
    if (argc != 2) {
        fprintf(stderr, "Usage: %s <machine_file.txt>\n", argv[0]);
        return -1;
    }
    if (load_machine(argv[1]) != 0) return -1;
    printf("Loaded machine: %d states, %d input letters, init = %d\n",
           NUM_STATE, NUM_INPUTLETTERS, INITSTATE);
    mealymachine(INITSTATE);
    return 0;
}


In [ ]:
# Compile and run with the 1-clock-delay machine.
# Pipe an input string into the binary so we can see deterministic output.
!gcc -Wall -o FA-mealy-file FA-mealy-file.c
!echo "00110110" | ./FA-mealy-file mealy1clk.txt


(Replace this line with the YouTube Shorts URL of your hand-recorded explanation. Script for the narration is in the chat.)

URLをここに記載．

**Goal.** Build a Mealy machine that outputs each input symbol *two* clocks later. The first two outputs default to `0` because there is no input from "2 steps ago" yet.

**State design.** The machine needs to remember "the last two input symbols seen so far". With input alphabet $\Sigma = \{0, 1\}$, there are $2 \times 2 = 4$ possible "last-two" pairs, plus a few startup states:

- **state 0** — `S_start`: no input received yet.
- **state 1** — `S_0?`: only one input received, and it was `0`.
- **state 2** — `S_1?`: only one input received, and it was `1`.
- **state 3** — `S_00`: last two inputs were `0, 0`.
- **state 4** — `S_01`: last two inputs were `0, 1`.
- **state 5** — `S_10`: last two inputs were `1, 0`.
- **state 6** — `S_11`: last two inputs were `1, 1`.

Total: **7 states**, 2 input letters.

**Transition rule.** From a state encoding `(prev, curr)` (i.e. `S_{prev}{curr}`), reading new input `x`:
- New state encodes `(curr, x)`.
- **Output is `prev`** — the symbol from 2 clocks ago, which is being "shifted out".

For the startup states (no `prev` yet), the output is `0` and we transition into the next startup or full state as appropriate.

**State-transition diagram (ASCII).**

```
                     input/output
   S_start ──0/0──> S_0?
   S_start ──1/0──> S_1?

   S_0?    ──0/0──> S_00
   S_0?    ──1/0──> S_01
   S_1?    ──0/0──> S_10
   S_1?    ──1/0──> S_11

   S_00 ──0/0──> S_00      S_00 ──1/0──> S_01
   S_01 ──0/0──> S_10      S_01 ──1/0──> S_11
   S_10 ──0/1──> S_00      S_10 ──1/1──> S_01
   S_11 ──0/1──> S_10      S_11 ──1/1──> S_11
```

**Tables.**

`ttable[state][input]` — next state on input 0 / input 1:
- state 0 (S_start): → 1, 2
- state 1 (S_0?):    → 3, 4
- state 2 (S_1?):    → 5, 6
- state 3 (S_00):    → 3, 4
- state 4 (S_01):    → 5, 6
- state 5 (S_10):    → 3, 4
- state 6 (S_11):    → 5, 6

`otable[state][input]` — output on input 0 / input 1:
- state 0 (S_start): 0, 0     (no "2 ago" yet)
- state 1 (S_0?):    0, 0     (still no "2 ago")
- state 2 (S_1?):    0, 0     (still no "2 ago")
- state 3 (S_00):    0, 0     ("2 ago" is 0)
- state 4 (S_01):    0, 0     ("2 ago" is 0)
- state 5 (S_10):    1, 1     ("2 ago" is 1)
- state 6 (S_11):    1, 1     ("2 ago" is 1)

**Sanity check.** Take input `00110110`. The expected 2-clock-delayed output is two zeros prepended and the input shifted right by 2: `00 001101`. Tracing through the table gives `0 0 0 0 1 1 0 1` — matches.

The next code cell implements this by only rewriting the machine-definition section of `FA-mealy.c` (lines 9–32), exactly as the assignment instructs.

In [ ]:
%%writefile FA-mealy-2clk.c
// Mealy machine: 2-clock-delay output (Task 10X).
// Only the machine-definition section (lines 9-32 area) is changed from
// the original FA-mealy.c. The execution loop (mealymachine + main)
// is identical to the original.

#include <stdio.h>

// -------------------------------
// 2-clock-delay machine definition
//
// 7 states:
//   0 = S_start  (no input received yet)
//   1 = S_0?     (one input, it was 0)
//   2 = S_1?     (one input, it was 1)
//   3 = S_00     (last two inputs: 0,0)
//   4 = S_01     (last two inputs: 0,1)
//   5 = S_10     (last two inputs: 1,0)
//   6 = S_11     (last two inputs: 1,1)
//
// From a state (prev, curr), reading input x:
//   next state = (curr, x)
//   output     = prev  (the symbol from 2 clocks ago, being shifted out)

#define NUM_STATE        7
#define NUM_INPUTLETTERS 2

#define INITSTATE 0

// transition table: ttable[current_state][input_letter] = next_state
int ttable[NUM_STATE][NUM_INPUTLETTERS] = {
  {1, 2}, // 0 S_start  -> S_0?, S_1?
  {3, 4}, // 1 S_0?     -> S_00, S_01
  {5, 6}, // 2 S_1?     -> S_10, S_11
  {3, 4}, // 3 S_00     -> S_00, S_01
  {5, 6}, // 4 S_01     -> S_10, S_11
  {3, 4}, // 5 S_10     -> S_00, S_01
  {5, 6}  // 6 S_11     -> S_10, S_11
};

// output table: otable[current_state][input_letter] = output_symbol
int otable[NUM_STATE][NUM_INPUTLETTERS] = {
  {0, 0}, // 0 S_start: nothing from "2 ago" yet
  {0, 0}, // 1 S_0?:    still nothing from "2 ago"
  {0, 0}, // 2 S_1?:    still nothing from "2 ago"
  {0, 0}, // 3 S_00:    "2 ago" was 0
  {0, 0}, // 4 S_01:    "2 ago" was 0
  {1, 1}, // 5 S_10:    "2 ago" was 1
  {1, 1}  // 6 S_11:    "2 ago" was 1
};
// -------------------------------

// Main body of Mealy machine (unchanged from FA-mealy.c)
int mealymachine(int ss){
  int c, t, cs, ns;
  printf("Input Output    : 1st  input => ");
  cs = ss;
  while ((c = getc(stdin)) != EOF) {
    t = c - '0';
    if (t >= 0 && t < NUM_INPUTLETTERS) {
      ns = ttable[cs][t];
      printf("    %d      %d    : next input => ", t, otable[cs][t]);
      cs = ns;
    }
  }
  return 0;
}

int main(int argc, char *argv[]){
  mealymachine(INITSTATE);
  return 0;
}


In [ ]:
# Compile and test the 2-clock-delay machine.
# Input "00110110" should produce output "00001101" (input shifted right by 2).
!gcc -Wall -o FA-mealy-2clk FA-mealy-2clk.c
!echo "00110110" | ./FA-mealy-2clk


In [ ]:
プログラムコード．

---
# 課題提出法

筑波大学工学システム学類３年生向け．
FG24711 / FG34711．

必須課題は全て実施すること．
発展課題はしなくともよいが，A+取得には発展課題を（全課題提出を通して）1つ以上実施していることが必要条件である．

該当するmanabaに課題提出のエントリを設けるので，そこに本**Colab notebookを提出**すること．



# 出典

筑波大学工学システム学類  
データ構造とアルゴリズム  
担当：亀田能成  


---
2026/06/10 ver.A  
